In [20]:
#import the pdf:
# for local file
from magic_doc.docconv import DocConverter
converter = DocConverter(s3_config=None)
markdown_content, time_cost = converter.convert("/Users/lijou/Documents/Documents/Project/Notes_Docs/PreAward/exemples/Binder1.pdf", conv_timeout=300)

print(markdown_content)


2024-08-01 15:05:16.059 | INFO     | magic_pdf.libs.pdf_check:detect_invalid_chars:57 - cid_count: 0, text_len: 43550, cid_chars_radio: 0.0
2024-08-01 15:05:16.133 | INFO     | magic_doc.contrib.pdf.pdf_extractor:run:70 - stream io data is digital pdf


Call: [HORIZON-CL6-2024-FARM2FORK-01-1] — [Agro-pastoral/outdoor livestock systems and wildlife management] 

Part B - Page 1 of 50 

COHABITATION AND OPTIMAL RECONCILIATION: INTEGRATING TERRITORIAL TRANSFORMATIONS AND ECOLOGICAL RESILIENCE

CRITTER  

[This document is tagged. Do not delete the tags; they are needed for processing.] #@APP-FORM-HERIAIA@# List of participants

PARTICIPANT NO. PARTICIPANT ORGANISATION NAME SHORT NAME COUNTRY

1 (Coordinator) Syddansk Universitet SDU DK

2 Lapin Yliopisto UOL FI

3 Università degli Studi di Padova UNIPD IT

4 Centro de Investigacion y Tecnologia Agroalimentaria de Aragon CITA ES

5 Association WWF Bulgaria WWF-BG BG

6 WWF Slovensko WWF-SK SK

7 Zavod za Gozdove Slovenije - Slovenia Forest Service SFS SI

8 Schola Campesina APS CAMP IT

9 De Surdurulebilir Enerji ve Insaat Sanayi Ticaret Limited Sirketi DEM TR

10 Luonnonvarakeskus - Natural Resources Institute Finland LUKE FI

11 Interspread GmbH INSP AT

12 Wageningen University & Resea

In [6]:
import ollama
from nltk import word_tokenize
import re
import math

# Define the refined personalities and their corresponding parameters
personalities_parameters = {
    'Highly analytical evaluator': {
        'model': 'mistral-nemo', # 'Qwen2', 'internlm2', 'Mathstral'
        'temperature': 0.3,
        'top_p': 0.8,
        'frequency_penalty': 1.5,
        'presence_penalty': 1.2
    },
    'Collaboration expert': {
        'model': 'mistral-nemo',
        'temperature': 0.6,
        'top_p': 0.9,
        'frequency_penalty': 1.0,
        'presence_penalty': 1.0
    },
    'Innovation and impact specialist': {
        'model': 'mistral-nemo',
        'temperature': 0.8,
        'top_p': 0.95,
        'frequency_penalty': 0.8,
        'presence_penalty': 1.0
    },
    'Project management expert': {
        'model': 'mistral-nemo',
        'temperature': 0.5,
        'top_p': 0.9,
        'frequency_penalty': 1.0,
        'presence_penalty': 1.0
    }
}

# Define the section-specific and complex questions
questions = {
    "Introduction and Excellence": {
        'Highly analytical evaluator': "Does the document present clear, measurable, and verifiable objectives? Is the proposed methodology sound, with all underlying concepts, models, and assumptions well-explained and logically consistent?",
        'Collaboration expert': "Does the methodology involve appropriate interdisciplinary approaches and demonstrate effective engagement with all relevant stakeholders, including examples of successful past collaborations or plans for future partnerships?",
        'Innovation and impact specialist': "Does the document clearly highlight how the project goes beyond the state-of-the-art with innovative concepts, approaches, and methodologies, and provide realistic and achievable objectives that could generate significant impact?",
        'Project management expert': "Does the work plan provide a detailed and efficient structure, including clear objectives, timelines, and resource allocation, with well-defined milestones and deliverables that ensure effective implementation?"
    },
    "Impact": {
        'Highly analytical evaluator': "Are the pathways to achieve the expected outcomes and impacts well-defined and credible? Are the measures to maximize these impacts suitable, realistic, and well-described with quantifiable estimates?",
        'Collaboration expert': "Does the document provide detailed plans for stakeholder engagement, dissemination, and exploitation activities that ensure broad uptake and impact of the project results among relevant target groups?",
        'Innovation and impact specialist': "Does the project have the potential to drive significant scientific, economic, and societal impacts? Are the expected outcomes and impacts clearly described, with an emphasis on innovation and the generation of new opportunities?",
        'Project management expert': "Are the dissemination, exploitation, and communication plans detailed and well-structured, with clear objectives, target groups, and measures to ensure effective communication and uptake of the project results?"
    },
    "Quality and Efficiency of the Implementation": {
        'Highly analytical evaluator': "Is the work plan detailed and efficient, with appropriate allocation of resources and efforts to each work package? Are risks identified and mitigation measures well-defined and realistic?",
        'Collaboration expert': "Does the consortium have the necessary expertise and infrastructure to achieve the project’s objectives? How well do the members complement each other and cover the value chain, ensuring effective collaboration and resource utilization?",
        'Innovation and impact specialist': "Does the project demonstrate robust innovation management and risk mitigation strategies? Are potential barriers identified, and are the proposed mitigation measures adequate to ensure successful implementation and impact of the project?",
        'Project management expert': "Does the work plan provide a comprehensive overview with clear roles and responsibilities for each participant? Are the timelines, milestones, and deliverables well-defined and aligned with the project objectives to ensure efficient implementation and monitoring?"
    }
}

def split_document(text):
    sections = re.split(r'(?i)2\. impact', text)
    if len(sections) > 1:
        sections[1] = '2. Impact' + sections[1]
    else:
        raise ValueError("Section '2. Impact' not found in the document.")
    
    sections = [re.split(r'(?i)3\. quality and efficiency of the implementation', section) for section in sections]
    flattened_sections = [item for sublist in sections for item in sublist]
    if len(flattened_sections) == 3:
        flattened_sections[2] = '3. Quality And Efficiency Of The Implementation' + flattened_sections[2]
    else:
        raise ValueError("Section '3. Quality And Efficiency Of The Implementation' not found in the document.")
    
    return flattened_sections

def calculate_context_window(sections):
    section_word_counts = [len(word_tokenize(section)) for section in sections]
    max_word_count = max(section_word_counts)
    context_window = int(max_word_count * 1.2)
    context_window = math.ceil(context_window / 10000) * 10000
    return context_window

def analyze_section(personality_key, params, section_title, section, context_window):
    question = questions[section_title][personality_key]
    messages = [
        {
            'role': 'system',
            'content': (
                f'You are {personality_key}, an expert evaluating an EU grant application. '
                'Your audience includes researchers and research support officers. '
                'The document consists of three main sections: "Introduction and Excellence," "Impact," and "Quality and Efficiency of the Implementation." '
                f'Use the following section of the document, titled "{section_title}", to answer the question. '
                'Follow these steps in your response:\n'
                '1. Provide a detailed analysis, including specific strengths and weaknesses of the section. List each strength and weakness as a separate bullet point and include as many as you find relevant.\n'
                '2. Offer actionable recommendations for improvement, with each recommendation as a separate bullet point.\n'
                '3. Conclude with a summary of your findings.\n\n'
                'Example Response Format:\n'
                f'Section {section_title}:\n'
                '1. **Strengths**:\n'
                '   - [Detail each strength]\n'
                '2. **Weaknesses**:\n'
                '   - [Detail each weakness]\n'
                '3. **Recommendations**:\n'
                '   - [Detail each recommendation]\n'
                '4. **Summary**:\n'
                '   - [Brief summary of the findings]\n\n'
                f'Document: {section}'
            ).replace("{section_title}", section_title),
        },
        {
            'role': 'user',
            'content': question,
        },
    ]
    options = {
        "num_ctx": context_window,
        "temperature": params['temperature'],
        "top_p": params['top_p'],
        "frequency_penalty": params['frequency_penalty'],
        "presence_penalty": params['presence_penalty']
    }
    response = ollama.chat(model=params['model'], messages=messages, options=options)
    response_content = response['message']['content']
    return response_content

def evaluate_all_experts(document_text):
    sections = split_document(document_text)
    context_window = calculate_context_window(sections)
    section_titles = [
        "Introduction and Excellence",
        "Impact",
        "Quality and Efficiency of the Implementation"
    ]

    all_expert_responses = []

    for personality_key, params in personalities_parameters.items():
        expert_responses = {"personality": personality_key, "responses": {}}
        for section_title, section in zip(section_titles, sections):
            print(f"The {personality_key} is analyzing the {section_title} section")
            answer = analyze_section(personality_key, params, section_title, section, context_window)
            expert_responses["responses"][section_title] = answer
        all_expert_responses.append(expert_responses)
    
    return all_expert_responses



def analyze_reviewer(section_title, consolidated_feedback):
    context_window = 5000
    messages = [
        {
            'role': 'system',
            'content': (
                'You are Comprehensive Reviewer, an expert tasked with synthesizing feedback from multiple expert reviews of an EU grant application. '
                'Your audience includes researchers and research support officers. '
                'The document consists of three main sections: "Introduction and Excellence," "Impact," and "Quality and Efficiency of the Implementation." '
                f'Use the consolidated feedback for the section titled "{section_title}" to provide a synthesis. '
                'Follow these steps in your response:\n'
                '1. Summarize the key strengths identified by the experts.\n'
                '2. Summarize the key weaknesses identified by the experts.\n'
                '3. Provide actionable recommendations for improvement based on the experts’ feedback.\n'
                '4. Conclude with a summary of your findings.\n\n'
                'Example Response Format:\n'
                f'Section {section_title}:\n'
                '1. **Strengths**:\n'
                '   - [Detail each strength]\n'
                '2. **Weaknesses**:\n'
                '   - [Detail each weakness]\n'
                '3. **Recommendations**:\n'
                '   - [Detail each recommendation]\n'
                '4. **Summary**:\n'
                '   - [Brief summary of the findings]\n\n'
                f'Consolidated Feedback: {consolidated_feedback[section_title]}'
            ).replace("{section_title}", section_title),
        },
        {
            'role': 'user',
            'content': (
                f'Please provide a comprehensive synthesis for the section titled "{section_title}" based on the consolidated feedback provided.'
            ),
        },
    ]
    options = {
        "num_ctx": context_window,
        "temperature": 0.5,
        "top_p": 0.9,
        "frequency_penalty": 1.0,
        "presence_penalty": 1.0
    }
    response = ollama.chat(model="mistral-nemo", messages=messages, options=options)
    response_content = response['message']['content']
    return response_content


def flatten_responses(all_expert_responses):
    flat_expert_responses = {}

    for expert in all_expert_responses:
        for section, response in expert["responses"].items():
            if section not in flat_expert_responses:
                flat_expert_responses[section] = []
            flat_expert_responses[section].append(f"Personality: {expert['personality']}\nResponse:\n{response}\n\n")

    # Convert the dictionary to a string
    consolidated_feedback_str = ""
    for section, responses in flat_expert_responses.items():
        consolidated_feedback_str += f"## {section}\n\n"
        consolidated_feedback_str += "\n".join(responses)
        consolidated_feedback_str += "\n\n"



def combine_reviews(final_review, combined_review):
    # Combine the reviews into a single string
    combined_content = []

    combined_content.append("# Final Review and Combined Review\n\n")

    # Append final review content
    combined_content.append("## Final Review\n")
    combined_content.append(final_review)
    combined_content.append("\n\n")

    # Append combined review content
    combined_content.append("## Combined Review\n")
    combined_content.append(combined_review)
    combined_content.append("\n")

    return "\n".join(combined_content)

def save_to_markdown(content, filepath):
    # Save the combined content to a markdown file
    with open(filepath, "w", encoding="utf-8") as file:
        file.write(content)


document_text = markdown_content  # Assign your document content here
all_expert_responses = evaluate_all_experts(document_text)
combined_review = flatten_responses(all_expert_responses)
final_review = []

sections = ["Introduction and Excellence", "Impact", "Quality and Efficiency of the Implementation"]
for section in sections:
    final_review.append(analyze_reviewer(section, combined_review))

#print("\n\n".join(final_review))

# Assume final_review and combined_review are already defined and contain the respective review content
final_review_text = "\n\n".join(final_review)
combined_review_text = combined_review  # If combined_review is already a single string

# Combine the reviews
combined_content = combine_reviews(final_review_text, combined_review_text)

# Define the path where the file will be saved
output_path = "/Users/lijou/Documents/Documents/Project/Notes_Docs/PreAward/exemples/final_combined_review.md"

# Save to markdown
save_to_markdown(combined_content, output_path)

print(f"Combined review saved to {os.path.abspath(output_path)}")


The Highly analytical evaluator is analyzing the Introduction and Excellence section
The Highly analytical evaluator is analyzing the Impact section
The Highly analytical evaluator is analyzing the Quality and Efficiency of the Implementation section
The Collaboration expert is analyzing the Introduction and Excellence section
The Collaboration expert is analyzing the Impact section
The Collaboration expert is analyzing the Quality and Efficiency of the Implementation section
The Innovation and impact specialist is analyzing the Introduction and Excellence section
The Innovation and impact specialist is analyzing the Impact section
The Innovation and impact specialist is analyzing the Quality and Efficiency of the Implementation section
The Project management expert is analyzing the Introduction and Excellence section
The Project management expert is analyzing the Impact section
The Project management expert is analyzing the Quality and Efficiency of the Implementation section


TypeError: sequence item 5: expected str instance, dict found

In [8]:
def flatten_responses(all_expert_responses):
    flat_expert_responses = {}

    for expert in all_expert_responses:
        for section, response in expert["responses"].items():
            if section not in flat_expert_responses:
                flat_expert_responses[section] = []
            flat_expert_responses[section].append(f"Personality: {expert['personality']}\nResponse:\n{response}\n\n")

    # Convert the dictionary to a string
    consolidated_feedback_str = ""
    for section, responses in flat_expert_responses.items():
        consolidated_feedback_str += f"## {section}\n\n"
        consolidated_feedback_str += "\n".join(responses)
        consolidated_feedback_str += "\n\n"
combined_review = flatten_responses(all_expert_responses)
final_review = []

sections = ["Introduction and Excellence", "Impact", "Quality and Efficiency of the Implementation"]
for section in sections:
    final_review.append(analyze_reviewer(section, consolidated_feedback))

#print("\n\n".join(final_review))

# Assume final_review and combined_review are already defined and contain the respective review content
final_review_text = "\n\n".join(final_review)
combined_review_text = combined_review  # If combined_review is already a single string

# Combine the reviews
combined_content = combine_reviews(final_review_text, combined_review_text)

# Define the path where the file will be saved
output_path = "/Users/lijou/Documents/Documents/Project/Notes_Docs/PreAward/exemples/final_combined_review.md"

# Save to markdown
save_to_markdown(combined_content, output_path)

print(f"Combined review saved to {os.path.abspath(output_path)}")


TypeError: sequence item 5: expected str instance, dict found

In [3]:
# print the responses of all experts in an easy-to-read format as a text, not using flatted_responses
for expert in all_expert_responses:
    print(f"Personality: {expert['personality']}\n")
    for section, response in expert["responses"].items():
        print(f"Section: {section}\n")
        print(f"Response:\n{response}\n")
    print("="*50 + "\n")


Personality: Highly analytical evaluator

Section: Introduction and Excellence

Response:
To evaluate whether a document presents clear, measurable, verifiable objectives along with a sound methodology that is well-explained and logically consistent involves several steps. Here's how you might approach this:

1. **Objectives:**
   - Check if the goals are explicitly stated at the beginning of the document.
   - Ensure they are specific (not vague or broad), measurable (quantifiable where possible), achievable, relevant, time-bound (SMART criteria).
   - Verify that these objectives can be tracked and assessed throughout the project lifecycle.

2. **Methodology:**
   - Review if a clear methodology is outlined with sufficient detail to understand how it will achieve the stated objectives.
   - Look for explanations of underlying concepts: Are they well-defined, relevant, and appropriate?
   - Check models used:
     * Do they fit within accepted standards or best practices in their resp

In [ ]:
consolidated_feedback = flatten_responses(all_expert_responses)
final_review = []

sections = ["Introduction and Excellence", "Impact", "Quality and Efficiency of the Implementation"]
for section in sections:
    final_review.append(analyze_reviewer(section, consolidated_feedback))

#print("\n\n".join(final_review))
# Assume final_review and combined_review are already defined and contain the respective review content
final_review_text = "\n\n".join(final_review)
combined_review_text = combined_review  # If combined_review is already a single string

# Combine the reviews
combined_content = combine_reviews(final_review_text, combined_review_text)

# Define the path where the file will be saved
output_path = "/Users/lijou/Documents/Documents/Project/Notes_Docs/PreAward/exemples/final_combined_review.md"

# Save to markdown
save_to_markdown(combined_content, output_path)

print(f"Combined review saved to {os.path.abspath(output_path)}")

In [53]:
def analyze_reviewer(section_title, consolidated_feedback):
    context_window = 5000
    messages = [
        {
            'role': 'system',
            'content': (
                'You are Comprehensive Reviewer, an expert tasked with synthesizing feedback from multiple expert reviews of an EU grant application. '
                'Your audience includes researchers and research support officers. '
                'The document consists of three main sections: "Introduction and Excellence," "Impact," and "Quality and Efficiency of the Implementation." '
                f'Use the consolidated feedback for the section titled "{section_title}" to provide a synthesis. '
                'Follow these steps in your response:\n'
                '1. Summarize the key strengths identified by the experts.\n'
                '2. Summarize the key weaknesses identified by the experts.\n'
                '3. Provide actionable recommendations for improvement based on the experts’ feedback.\n'
                '4. Conclude with a summary of your findings.\n\n'
                'Example Response Format:\n'
                f'Section {section_title}:\n'
                '1. **Strengths**:\n'
                '   - [Detail each strength]\n'
                '2. **Weaknesses**:\n'
                '   - [Detail each weakness]\n'
                '3. **Recommendations**:\n'
                '   - [Detail each recommendation]\n'
                '4. **Summary**:\n'
                '   - [Brief summary of the findings]\n\n'
                f'Consolidated Feedback:\n'
                f'{consolidated_feedback[section_title]}'
            ),
        },
        {
            'role': 'user',
            'content': (
                f'Please provide a comprehensive synthesis for the section titled "{section_title}" based on the consolidated feedback provided.'
            ),
        },
    ]
    options = {
        "num_ctx": context_window,
        "temperature": 0.5,
        "top_p": 0.9,
        "frequency_penalty": 1.0,
        "presence_penalty": 1.0
    }
    response = ollama.chat(model="llama3-groq-tool-use", messages=messages, options=options)
    response_content = response['message']['content']
    return response_content


In [55]:
sections = ["Introduction and Excellence", "Impact", "Quality and Efficiency of the Implementation"]

# Create a dictionary to store the consolidated feedback
consolidated_feedback = {section: "" for section in sections}

# Iterate over each section
for section in sections:
    # For each section, iterate over each expert
    for expert in all_expert_responses:
        # Extract the expert's feedback for the current section
        feedback = expert['responses'][section]
        # Add this feedback to the consolidated feedback for the current section,
        # including the expert's personality
        consolidated_feedback[section] += f"Expert: {expert['personality']}\n{feedback}\n\n"

# Now you can use the consolidated feedback to generate the final review
final_review = []
for section in sections:
    final_review.append(analyze_reviewer(section, consolidated_feedback[section]))



TypeError: string indices must be integers, not 'str'

In [43]:

def parse_feedback(text):
    # Initialize dictionary to store parsed feedback
    feedback_dict = {}
    
    # Split the text into sections by personality
    sections = text.split("==================================================")
    
    for section in sections:
        if section.strip():  # Check if the section is not empty
            # Extract personality
            personality_match = re.search(r'Personality: (.+)', section)
            if personality_match:
                personality = personality_match.group(1).strip()
                feedback_dict[personality] = {}
                
                # Extract sections and their responses
                section_matches = re.findall(r'Section: ([\w\s]+)\nResponse:\n(.+?)(?=\nSection:|\Z)', section, re.DOTALL)
                for section_title, response in section_matches:
                    feedback_dict[personality][section_title.strip()] = response.strip()
    
    return feedback_dict


consolidated_feedback_dict = parse_feedback(consolidated_feedback)

In [48]:

# print all_expert_responses in a more readable format
for expert in all_expert_responses:
    print(f"Personality: {expert['personality']}\n")
    for section, response in expert["responses"].items():
        print(f"Section: {section}\n")
        print(f"Response:\n{response}\n")
    print("="*50 + "\n")

Personality: Highly analytical evaluator

Section: Introduction and Excellence

Response:
1. **Clear Objectives**: The paper should clearly state its research questions or hypotheses at the outset. These must be specific enough to guide your work but broad enough that they can accommodate a range of findings (i.e., not so narrow as to preclude any results).
   - Example: "The aim of this study is to determine whether there's an association between regular exercise and mental health in young adults."
   
2. **Measurable Objectives**: The objectives should be quantifiable, meaning you can measure progress towards them using data or other evidence. This helps ensure that the research process remains focused on achieving these goals rather than drifting off-topic as it proceeds (i.e., not becoming unfocused).
   - Example: "We will collect survey responses from a sample of young adults to assess their exercise habits and mental health."
   
3. **Verifiable Objectives**: The objectives shou

In [46]:
# print the consolidated_feedback_dict in a more readable format
import json
print(json.dumps(consolidated_feedback_dict, indent=4))

{
    "Highly analytical evaluator": {
        "Introduction and Excellence": "1. **Clear Objectives**: The paper should clearly state its research questions or hypotheses at the outset. These must be specific enough to guide your work but broad enough that they can accommodate a range of findings (i.e., not so narrow as to preclude any results).\n   - Example: \"The aim of this study is to determine whether there's an association between regular exercise and mental health in young adults.\"\n   \n2. **Measurable Objectives**: The objectives should be quantifiable, meaning you can measure progress towards them using data or other evidence. This helps ensure that the research process remains focused on achieving these goals rather than drifting off-topic as it proceeds (i.e., not becoming unfocused).\n   - Example: \"We will collect survey responses from a sample of young adults to assess their exercise habits and mental health.\"\n   \n3. **Verifiable Objectives**: The objectives shoul

In [45]:

sections = ["Introduction and Excellence", "Impact", "Quality and Efficiency of the Implementation"]
for section in sections:
    final_review.append(analyze_reviewer(section, consolidated_feedback_dict))

#print("\n\n".join(final_review))

# Assume final_review and combined_review are already defined and contain the respective review content
final_review_text = "\n\n".join(final_review)
combined_review_text = combined_review  # If combined_review is already a single string

# Combine the reviews
combined_content = combine_reviews(final_review_text, combined_review_text)

# Define the path where the file will be saved
output_path = "/Users/lijou/Documents/Documents/Project/Notes_Docs/PreAward/exemples/final_combined_review.md"

# Save to markdown
save_to_markdown(combined_content, output_path)

print(f"Combined review saved to {os.path.abspath(output_path)}")

KeyError: 'Introduction and Excellence'

In [39]:
import os

def combine_reviews(final_review, combined_review):
    # Combine the reviews into a single string
    combined_content = []

    combined_content.append("# Final Review and Combined Review\n\n")

    # Append final review content
    combined_content.append("## Final Review\n")
    combined_content.append(final_review)
    combined_content.append("\n\n")

    # Append combined review content
    combined_content.append("## Combined Review\n")
    combined_content.append(consolidated_feedback)
    combined_content.append("\n")

    return "\n".join(combined_content)

def save_to_markdown(content, filepath):
    # Save the combined content to a markdown file
    with open(filepath, "w", encoding="utf-8") as file:
        file.write(content)

# Assume final_review and combined_review are already defined and contain the respective review content
final_review_text = "\n\n".join(final_review)
combined_review_text = combined_review  # If combined_review is already a single string

# Combine the reviews
combined_content = combine_reviews(final_review_text, combined_review_text)

# Define the path where the file will be saved
output_path = "/Users/lijou/Documents/Documents/Project/Notes_Docs/PreAward/exemples/final_combined_review.md"

# Save to markdown
save_to_markdown(combined_content, output_path)

print(f"Combined review saved to {os.path.abspath(output_path)}")


Combined review saved to /Users/lijou/Documents/Documents/Project/Notes_Docs/PreAward/exemples/final_combined_review.md


In [42]:
print(consolidated_feedback)

Personality: Highly analytical evaluator

Section: Introduction and Excellence
Response:
1. **Clear Objectives**: The paper should clearly state its research questions or hypotheses at the outset. These must be specific enough to guide your work but broad enough that they can accommodate a range of findings (i.e., not so narrow as to preclude any results).
   - Example: "The aim of this study is to determine whether there's an association between regular exercise and mental health in young adults."
   
2. **Measurable Objectives**: The objectives should be quantifiable, meaning you can measure progress towards them using data or other evidence. This helps ensure that the research process remains focused on achieving these goals rather than drifting off-topic as it proceeds (i.e., not becoming unfocused).
   - Example: "We will collect survey responses from a sample of young adults to assess their exercise habits and mental health."
   
3. **Verifiable Objectives**: The objectives shoul

In [15]:
import ollama
from nltk import word_tokenize
import re
import math
import os

# Define the refined personalities and their corresponding parameters
personalities_parameters = {
    'Highly analytical evaluator': {
        'model': 'mistral-nemo', # 'Qwen2', 'internlm2', 'Mathstral'
        'temperature': 0.3,
        'top_p': 0.8,
        'frequency_penalty': 1.5,
        'presence_penalty': 1.2
    },
    'Collaboration expert': {
        'model': 'mistral-nemo',
        'temperature': 0.6,
        'top_p': 0.9,
        'frequency_penalty': 1.0,
        'presence_penalty': 1.0
    },
    'Innovation and impact specialist': {
        'model': 'mistral-nemo',
        'temperature': 0.8,
        'top_p': 0.95,
        'frequency_penalty': 0.8,
        'presence_penalty': 1.0
    },
    'Project management expert': {
        'model': 'mistral-nemo',
        'temperature': 0.5,
        'top_p': 0.9,
        'frequency_penalty': 1.0,
        'presence_penalty': 1.0
    }
}

# Define the section-specific and complex questions
questions = {
    "Introduction and Excellence": {
        'Highly analytical evaluator': "Does the document present clear, measurable, and verifiable objectives? Is the proposed methodology sound, with all underlying concepts, models, and assumptions well-explained and logically consistent?",
        'Collaboration expert': "Does the methodology involve appropriate interdisciplinary approaches and demonstrate effective engagement with all relevant stakeholders, including examples of successful past collaborations or plans for future partnerships?",
        'Innovation and impact specialist': "Does the document clearly highlight how the project goes beyond the state-of-the-art with innovative concepts, approaches, and methodologies, and provide realistic and achievable objectives that could generate significant impact?",
        'Project management expert': "Does the work plan provide a detailed and efficient structure, including clear objectives, timelines, and resource allocation, with well-defined milestones and deliverables that ensure effective implementation?"
    },
    "Impact": {
        'Highly analytical evaluator': "Are the pathways to achieve the expected outcomes and impacts well-defined and credible? Are the measures to maximize these impacts suitable, realistic, and well-described with quantifiable estimates?",
        'Collaboration expert': "Does the document provide detailed plans for stakeholder engagement, dissemination, and exploitation activities that ensure broad uptake and impact of the project results among relevant target groups?",
        'Innovation and impact specialist': "Does the project have the potential to drive significant scientific, economic, and societal impacts? Are the expected outcomes and impacts clearly described, with an emphasis on innovation and the generation of new opportunities?",
        'Project management expert': "Are the dissemination, exploitation, and communication plans detailed and well-structured, with clear objectives, target groups, and measures to ensure effective communication and uptake of the project results?"
    },
    "Quality and Efficiency of the Implementation": {
        'Highly analytical evaluator': "Is the work plan detailed and efficient, with appropriate allocation of resources and efforts to each work package? Are risks identified and mitigation measures well-defined and realistic?",
        'Collaboration expert': "Does the consortium have the necessary expertise and infrastructure to achieve the project’s objectives? How well do the members complement each other and cover the value chain, ensuring effective collaboration and resource utilization?",
        'Innovation and impact specialist': "Does the project demonstrate robust innovation management and risk mitigation strategies? Are potential barriers identified, and are the proposed mitigation measures adequate to ensure successful implementation and impact of the project?",
        'Project management expert': "Does the work plan provide a comprehensive overview with clear roles and responsibilities for each participant? Are the timelines, milestones, and deliverables well-defined and aligned with the project objectives to ensure efficient implementation and monitoring?"
    }
}

def split_document(text):
    print("Splitting the document into sections...")
    sections = re.split(r'(?i)2\. impact', text)
    if len(sections) > 1:
        sections[1] = '2. Impact' + sections[1]
    else:
        raise ValueError("Section '2. Impact' not found in the document.")

    sections = [re.split(r'(?i)3\. quality and efficiency of the implementation', section) for section in sections]
    flattened_sections = [item for sublist in sections for item in sublist]
    if len(flattened_sections) == 3:
        flattened_sections[2] = '3. Quality And Efficiency Of The Implementation' + flattened_sections[2]
    else:
        raise ValueError("Section '3. Quality And Efficiency Of The Implementation' not found in the document.")

    print("Document split into sections successfully.")
    return flattened_sections

def calculate_context_window(sections):
    print("Calculating context window...")
    section_word_counts = [len(word_tokenize(section)) for section in sections]
    max_word_count = max(section_word_counts)
    context_window = int(max_word_count * 1.2)
    context_window = math.ceil(context_window / 10000) * 10000
    print(f"Context window calculated: {context_window}")
    return context_window

def analyze_section(personality_key, params, section_title, section, context_window):
    print(f"Analyzing section '{section_title}' with personality '{personality_key}'...")
    question = questions[section_title][personality_key]
    messages = [
        {
            'role': 'system',
            'content': (
                f'You are {personality_key}, an expert evaluating an EU grant application. '
                'Your audience includes researchers and research support officers. '
                'The document consists of three main sections: "Introduction and Excellence," "Impact," and "Quality and Efficiency of the Implementation." '
                f'Use the following section of the document, titled "{section_title}", to answer the question. '
                'Follow these steps in your response:\n'
                '1. Provide a detailed analysis, including specific strengths and weaknesses of the section. List each strength and weakness as a separate bullet point and include as many as you find relevant.\n'
                '2. Offer actionable recommendations for improvement, with each recommendation as a separate bullet point.\n'
                '3. Conclude with a summary of your findings.\n\n'
                'Example Response Format:\n'
                f'Section {section_title}:\n'
                '1. **Strengths**:\n'
                '   - [Detail each strength]\n'
                '2. **Weaknesses**:\n'
                '   - [Detail each weakness]\n'
                '3. **Recommendations**:\n'
                '   - [Detail each recommendation]\n'
                '4. **Summary**:\n'
                '   - [Brief summary of the findings]\n\n'
                f'Document: {section}'
            ).replace("{section_title}", section_title),
        },
        {
            'role': 'user',
            'content': question,
        },
    ]
    options = {
        "num_ctx": context_window,
        "temperature": params['temperature'],
        "top_p": params['top_p'],
        "frequency_penalty": params['frequency_penalty'],
        "presence_penalty": params['presence_penalty']
    }
    response = ollama.chat(model=params['model'], messages=messages, options=options)
    response_content = response['message']['content']
    print(f"Analysis for section '{section_title}' with personality '{personality_key}' completed.")
    return response_content

def evaluate_all_experts(document_text):
    print("Evaluating all experts...")
    sections = split_document(document_text)
    context_window = calculate_context_window(sections)
    section_titles = [
        "Introduction and Excellence",
        "Impact",
        "Quality and Efficiency of the Implementation"
    ]

    all_expert_responses = []

    for personality_key, params in personalities_parameters.items():
        expert_responses = {"personality": personality_key, "responses": {}}
        for section_title, section in zip(section_titles, sections):
            print(f"The {personality_key} is analyzing the {section_title} section")
            answer = analyze_section(personality_key, params, section_title, section, context_window)
            expert_responses["responses"][section_title] = answer
        all_expert_responses.append(expert_responses)

    print("All experts evaluated successfully.")
    return all_expert_responses

def analyze_reviewer(section_title, consolidated_feedback):
    print(f"Analyzing reviewer for section '{section_title}'...")
    context_window = 5000
    messages = [
        {
            'role': 'system',
            'content': (
                'You are Comprehensive Reviewer, an expert tasked with synthesizing feedback from multiple expert reviews of an EU grant application. '
                'Your audience includes researchers and research support officers. '
                'The document consists of three main sections: "Introduction and Excellence," "Impact," and "Quality and Efficiency of the Implementation." '
                f'Use the consolidated feedback for the section titled "{section_title}" to provide a synthesis. '
                'Follow these steps in your response:\n'
                '1. Summarize the key strengths identified by the experts.\n'
                '2. Summarize the key weaknesses identified by the experts.\n'
                '3. Provide actionable recommendations for improvement based on the experts’ feedback.\n'
                '4. Conclude with a summary of your findings.\n\n'
                'Example Response Format:\n'
                f'Section {section_title}:\n'
                '1. **Strengths**:\n'
                '   - [Detail each strength]\n'
                '2. **Weaknesses**:\n'
                '   - [Detail each weakness]\n'
                '3. **Recommendations**:\n'
                '   - [Detail each recommendation]\n'
                '4. **Summary**:\n'
                '   - [Brief summary of the findings]\n\n'
                f'Consolidated Feedback: {consolidated_feedback[section_title]}'
            ).replace("{section_title}", section_title),
        },
        {
            'role': 'user',
            'content': (
                f'Please provide a comprehensive synthesis for the section titled "{section_title}" based on the consolidated feedback provided.'
            ),
        },
    ]
    options = {
        "num_ctx": context_window,
        "temperature": 0.5,
        "top_p": 0.9,
        "frequency_penalty": 1.0,
        "presence_penalty": 1.0
    }
    response = ollama.chat(model="mistral-nemo", messages=messages, options=options)
    response_content = response['message']['content']
    print(f"Reviewer analysis for section '{section_title}' completed.")
    return response_content

def flatten_responses(all_expert_responses):
    print("Flattening expert responses...")
    flat_expert_responses = {}

    for expert in all_expert_responses:
        for section, response in expert["responses"].items():
            if section not in flat_expert_responses:
                flat_expert_responses[section] = []
            flat_expert_responses[section].append(f"Personality: {expert['personality']}\nResponse:\n{response}\n\n")

    # Convert the dictionary to a string
    consolidated_feedback_str = ""
    for section, responses in flat_expert_responses.items():
        consolidated_feedback_str += f"## {section}\n\n"
        consolidated_feedback_str += "\n".join(responses)
        consolidated_feedback_str += "\n\n"

    print("Expert responses flattened successfully.")
    return consolidated_feedback_str

def combine_reviews(final_review, combined_review):
    print("Combining reviews...")
    # Combine the reviews into a single string
    combined_content = []

    combined_content.append("# Final Review and Combined Review\n\n")

    # Append final review content
    combined_content.append("## Final Review\n")
    combined_content.append(final_review)
    combined_content.append("\n\n")

    # Append combined review content
    combined_content.append("## Combined Review\n")
    combined_content.append(combined_review)
    combined_content.append("\n")

    print("Reviews combined successfully.")
    return "\n".join(combined_content)

def save_to_markdown(content, filepath):
    print(f"Saving combined content to {filepath}...")
    # Save the combined content to a markdown file
    with open(filepath, "w", encoding="utf-8") as file:
        file.write(content)
    print(f"Combined review saved to {os.path.abspath(filepath)}")

document_text = markdown_content  # Assign your document content here
print("Starting evaluation process...")
all_expert_responses = evaluate_all_experts(document_text)
combined_review = flatten_responses(all_expert_responses)
final_review = []

sections = ["Introduction and Excellence", "Impact", "Quality and Efficiency of the Implementation"]
for section in sections:
    final_review.append(analyze_reviewer(section, combined_review))

#print("\n\n".join(final_review))

# Assume final_review and combined_review are already defined and contain the respective review content
final_review_text = "\n\n".join(final_review)
combined_review_text = combined_review  # If combined_review is already a single string

# Combine the reviews
combined_content = combine_reviews(final_review_text, combined_review_text)

# Define the path where the file will be saved
output_path = "/Users/lijou/Documents/Documents/Project/Notes_Docs/PreAward/exemples/final_combined_review.md"

# Save to markdown
save_to_markdown(combined_content, output_path)

print("Evaluation process completed.")


Starting evaluation process...
Evaluating all experts...
Splitting the document into sections...
Document split into sections successfully.
Calculating context window...
Context window calculated: 20000
The Highly analytical evaluator is analyzing the Introduction and Excellence section
Analyzing section 'Introduction and Excellence' with personality 'Highly analytical evaluator'...
Analysis for section 'Introduction and Excellence' with personality 'Highly analytical evaluator' completed.
The Highly analytical evaluator is analyzing the Impact section
Analyzing section 'Impact' with personality 'Highly analytical evaluator'...
Analysis for section 'Impact' with personality 'Highly analytical evaluator' completed.
The Highly analytical evaluator is analyzing the Quality and Efficiency of the Implementation section
Analyzing section 'Quality and Efficiency of the Implementation' with personality 'Highly analytical evaluator'...
Analysis for section 'Quality and Efficiency of the Impleme

TypeError: string indices must be integers, not 'str'

In [17]:
print(all_expert_responses)

# print the responses of all experts in an easy-to-read format as a text, not using flatted_responses
#for expert in all_expert_responses:
#    print(f"Personality: {expert['personality']}\n")
#    for section, response in expert["responses"].items():
#        print(f"Section: {section}\n")
#        print(f"Response:\n{response}\n")
#    print("="*50 + "\n")

[{'personality': 'Highly analytical evaluator', 'responses': {'Introduction and Excellence': 'To evaluate whether a document presents clear, measurable, verifiable objectives along with a sound methodology that is clearly explained:\n\n**1. Objectives:**\n   - **Clear**: Are they stated explicitly at the beginning of the document?\n     *Example*: "The primary objective of this project is to increase sales by 20% within six months."\n   - **Measurable**: Can progress be tracked and success determined using quantitative metrics or indicators? If yes, specify them.\n     *Example*: Instead of \'improve customer satisfaction\', consider \'increase Net Promoter Score (NPS) from current score X to Y by the end of this quarter\'.\n   - **Verifiable**: Are there ways to confirm if these objectives have been achieved?\n     *Example*: For a project aiming at reducing waste, include methods like regular audits or comparing before-and-after data.\n\n**2. Methodology:**\n   - Is it well-explained

In [19]:
def analyze_reviewer(section_title, all_expert_responses):
    print(f"Analyzing reviewer for section '{section_title}'...")
    context_window = 5000

    # Extract the consolidated feedback for the given section
    consolidated_feedback = ""
    for expert in all_expert_responses:
        if section_title in expert["responses"]:
            consolidated_feedback += f"Personality: {expert['personality']}\nResponse:\n{expert['responses'][section_title]}\n\n"

    messages = [
        {
            'role': 'system',
            'content': (
                'You are Comprehensive Reviewer, an expert tasked with synthesizing feedback from multiple expert reviews of an EU grant application. '
                'Your audience includes researchers and research support officers. '
                'The document consists of three main sections: "Introduction and Excellence," "Impact," and "Quality and Efficiency of the Implementation." '
                f'Use the consolidated feedback for the section titled "{section_title}" to provide a synthesis. '
                'Follow these steps in your response:\n'
                '1. Summarize the key strengths identified by the experts.\n'
                '2. Summarize the key weaknesses identified by the experts.\n'
                '3. Provide actionable recommendations for improvement based on the experts’ feedback.\n'
                '4. Conclude with a summary of your findings.\n\n'
                'Example Response Format:\n'
                f'Section {section_title}:\n'
                '1. **Strengths**:\n'
                '   - [Detail each strength]\n'
                '2. **Weaknesses**:\n'
                '   - [Detail each weakness]\n'
                '3. **Recommendations**:\n'
                '   - [Detail each recommendation]\n'
                '4. **Summary**:\n'
                '   - [Brief summary of the findings]\n\n'
                f'Consolidated Feedback: {consolidated_feedback}'
            ).replace("{section_title}", section_title),
        },
        {
            'role': 'user',
            'content': (
                f'Please provide a comprehensive synthesis for the section titled "{section_title}" based on the consolidated feedback provided.'
            ),
        },
    ]
    options = {
        "num_ctx": context_window,
        "temperature": 0.5,
        "top_p": 0.9,
        "frequency_penalty": 1.0,
        "presence_penalty": 1.0
    }
    response = ollama.chat(model="mistral-nemo", messages=messages, options=options)
    response_content = response['message']['content']
    print(f"Reviewer analysis for section '{section_title}' completed.")
    return response_content

def combine_reviews(final_review, all_expert_responses):
    print("Combining reviews...")
    # Combine the reviews into a single string
    combined_content = []

    combined_content.append("# Final Review and Combined Review\n\n")

    # Append final review content
    combined_content.append("## Final Review\n")
    combined_content.append(final_review)
    combined_content.append("\n\n")

    # Append combined review content
    combined_content.append("## Combined Review\n")

    # Convert the dictionary to a string
    for expert in all_expert_responses:
        combined_content.append(f"### {expert['personality']}\n")
        for section, response in expert["responses"].items():
            combined_content.append(f"#### {section}\n")
            combined_content.append(response)
            combined_content.append("\n\n")

    combined_content.append("\n")

    print("Reviews combined successfully.")
    return "\n".join(combined_content)
 
final_review = []

sections = ["Introduction and Excellence", "Impact", "Quality and Efficiency of the Implementation"]
for section in sections:
    final_review.append(analyze_reviewer(section, all_expert_responses))

#print("\n\n".join(final_review))

# Assume final_review and combined_review are already defined and contain the respective review content
final_review_text = "\n\n".join(final_review)
combined_review_text = combined_review  # If combined_review is already a single string

# Combine the reviews
combined_content = combine_reviews(final_review_text, all_expert_responses)

# Define the path where the file will be saved
output_path = "/Users/lijou/Documents/Documents/Project/Notes_Docs/PreAward/exemples/final_combined_review.md"

# Save to markdown
save_to_markdown(combined_content, output_path)

print("Evaluation process completed.")

Analyzing reviewer for section 'Introduction and Excellence'...
Reviewer analysis for section 'Introduction and Excellence' completed.
Analyzing reviewer for section 'Impact'...
Reviewer analysis for section 'Impact' completed.
Analyzing reviewer for section 'Quality and Efficiency of the Implementation'...
Reviewer analysis for section 'Quality and Efficiency of the Implementation' completed.
Combining reviews...
Reviews combined successfully.
Saving combined content to /Users/lijou/Documents/Documents/Project/Notes_Docs/PreAward/exemples/final_combined_review.md...
Combined review saved to /Users/lijou/Documents/Documents/Project/Notes_Docs/PreAward/exemples/final_combined_review.md
Evaluation process completed.
